# Lab5 - Assignment 5 about extraction of properties (extra assignment)

Copyright, Vrije Universiteit Amsterdam, Faculty of Humanities, CLTL

This notebook describes the LAB-5 assignment of the Text Mining course. It is about Property Extraction.

**Assignment goals**:
* Get insight into the challenges of entity property extraction.
* Learn how to build a transparent property extraction method based on patterns.
* Get insight into the pros and cons of two pattern-based property extraction methods.
* Be able to run your extractors on unseen documents from Wikipedia.
* Be able to evaluate property extractors.

In this assignment, the main focus lies on creating your own pattern-based property extractors. You are then going to run them on Wikipedia texts, evaluate them against gold values, and reflect on their relative performance.

 We recommend that you go through the notebooks in the following order:
* *Read the assignment (see below)*
* *Lab5-Property-extraction.ipynb*
* *Answer the questions of the assignment (see below) using the provided notebooks and submit*

**Hint:** in the explanation notebook, we had an example about extraction of properties with substring matching and with dependencies. You can use much of that code here, but make sure you make the right adjustments.

**Good luck & have fun!**

### 1. Extracting properties with substring matching (12 points)

**Exercise 1a** Write code that extracts the birth year of a person by using substring matching. (4 points)


In [2]:
import re

def extract_birth_year_regex(sentence):
    # Extract the birth year of a person with regular expressions
    birth_year = re.search(r'\d{4}', sentence)

    return int(birth_year[0])

**Exercise 1b** Test your *birth year substring matching extractor* in the following way. 

* Write a sentence on which you expect that the extractor *WILL* work. 
* Write a sentence on which you expect that the extractor *WILL NOT* work. 

Run your extractor on both sentences and print the results. Make sure that the results are as expected. (2 points)

In [3]:
import lab5_utils as utils
import spacy
from spacy.util import filter_spans

nlp = spacy.load('en_core_web_sm')

def extract_birth_year_relations(doc, patterns):   
    property_value_type='DATE'
    target_entity_type='PERSON'
    
    # the following 3 lines merge entities and noun chunks into one token
    # The tokens "New" "York" will be retokenized into a single new token "New York"
    # Check out the spaCy documentation for details
    # The tokens in doc will be modified after calling this function
    # this is useful in our cases, so we will always do it.
    spans = list(doc.ents) + list(doc.noun_chunks)
    # I changed this part, previous code was not working
    with doc.retokenize() as retokenizer:
        for span in filter_spans(spans):
            retokenizer.merge(span) 
    
    relations = {}
    
    # step Ia - generate possible property values
    dates=utils.get_entities_of_type(property_value_type, doc)
    
    for date in dates:
        # step Ib - is one of our patterns found before the date 
        if utils.pattern_found_on_the_left(doc, date.i, patterns):
            # step II - find the closest entity of some target type
            person=utils.find_closest_entity(doc.ents, date.idx, target_entity_type)
            # step III - normalize the year
            year=extract_birth_year_regex(date.text)
            if year and person:
                relations[person]=year
    return relations

# Text where it won't work
text_bad = 'Julius was born on November 6th 1966.'
doc_bad = nlp(text_bad)

# Text where it will work
text_good = 'Julius was born in 1966.'
doc_good = nlp(text_good)

# The pattern(s)
founded_patterns = ['born in']

# Tests
date_relations_bad=extract_birth_year_relations(doc_bad, founded_patterns)
date_relations_good=extract_birth_year_relations(doc_good, founded_patterns)
print(f'NOT working: {text_bad} \nResult: {date_relations_bad}')
print(f'Working: {text_good} \nResult: {date_relations_good}')

NOT working: Julius was born on November 6th 1966. 
Result: {}
Working: Julius was born in 1966. 
Result: {'Julius': 1966}


**Exercise 1c** Write code that extracts the manufacturer of a device by using substring matching. (4 points)

In [10]:
def extract_manufacturer_regex(text):
    # Extract the manufacturer of a device by using regular expressions
    manufacturer = re.search('[A-Z][a-z]*', text)
    return manufacturer[0]

**Exercise 1d** Test your *manufacturer substring matching extractor* in the following way. 

* Write a sentence on which you expect that the extractor *WILL* work. 
* Write a sentence on which you expect that the extractor *WILL NOT* work. 

Run your extractor on both sentences and print the results. Make sure that the results are as expected. (2 points)

In [15]:
def extract_manufacturer_relations(doc, patterns, main_entity):   
    property_value_type='ORG'
    target_entity_type='PRODUCT '
    
    # the following 3 lines merge entities and noun chunks into one token
    # The tokens in doc will be modified after calling this function
    # this is useful in our cases, so we will always do it.
    spans = list(doc.ents) + list(doc.noun_chunks)
    # I changed this part, previous code was not working
    with doc.retokenize() as retokenizer:
        for span in filter_spans(spans):
            retokenizer.merge(span) 
    
    relations = {}
    
    # step Ia - generate possible property values
    manus=utils.get_entities_of_type(property_value_type, doc)
    
    for manu in manus:
        # step Ib - is one of our patterns found before the date 
        if utils.pattern_found_on_the_left(doc, manu.i, patterns):
            # step II - find the closest entity of some target type
            device=utils.find_closest_entity(doc.ents, manu.idx, target_entity_type)
            # if no device is found, assume relation is about main entity
            if not device:
                device=main_entity
            if device and manu:
                relations[device]=manu.text
    return relations


# Text where it won't work
text_bad = 'The iPhone, which was created by Steve Jobs, is very popular.'
doc_bad = nlp(text_bad)

# Text where it will work
text_good = 'The iPhone was made by Apple in 2007.'
doc_good = nlp(text_good)

# The pattern(s)
founded_patterns = ['made by', 'developed by', 'created by', 'manufactured by', 'produced by'] #designed by?
main_entity = 'iPhone'

# Tests
date_relations_bad=extract_manufacturer_relations(doc_bad, founded_patterns, main_entity)
date_relations_good=extract_manufacturer_relations(doc_good, founded_patterns, main_entity)
print(f'NOT working: {text_bad} \nResult: {date_relations_bad}')
print(f'Working: {text_good} \nResult: {date_relations_good}')

NOT working: The iPhone, which was created by Steve Jobs, is very popular. 
Result: {}
Working: The iPhone was made by Apple in 2007. 
Result: {'iPhone': 'Apple'}


### 2. Extracting properties by using dependency information (12 points)

**Exercise 2a** Write code that extracts the birth year of a person by using dependency information. (4 points)

In [3]:
def extract_birth_year_dep(text):
    # Extract the birth year of a person with dependencies
    return

**Exercise 2b** Test your *birth year dependency extractor* in the following way. 

* Write a sentence on which you expect that the extractor *WILL* work. 
* Write a sentence on which you expect that the extractor *WILL NOT* work. 

Run your extractor on both sentences and print the results. Make sure that the results are as expected. (2 points)

In [ ]:
# Your code here...

**Exercise 2c** Write code that extracts the manufacturer of a device by using dependency information. (4 points)

In [27]:
def extract_manufacturer_dep(text):
    # Extract the manufacturer of a device with dependencies
    return

**Exercise 2d** Test your *manufacturer dependency extractor* in the following way. 

* Write a sentence on which you expect that the extractor *WILL* work. 
* Write a sentence on which you expect that the extractor *WILL NOT* work. 

Run your extractor on both sentences and print the results. Make sure that the results are as expected. (2 points)

In [6]:
# Your code here...

### 3. Running and evaluating extractors on Wikipedia (8 points)

We will run our extractors on 50 documents about people and 50 documents about devices. We provide code to load the lists of entities and the gold values.

In [25]:
import json
with open("birthyears.json", 'rb') as f:
    gold_birthyears=json.load(f)
    wiki_people=list(gold_birthyears.keys())
    
with open("manufacturers.json", 'rb') as f:
    gold_manufacturers=json.load(f)
    wiki_devices=list(gold_countries.keys())

The lists `wiki_people` and `wiki_devices` contain the names of 50 people and 50 devices, respectively.

The dictionaries `gold_birthyears` and `gold_manufacturers` contain gold values for each of these entities.

We provide a function that evaluates your extracted property values against known ("gold") property values. The function returns three evaluation scores: precision, recall, f1-score. You can find call this function as follows:

`utils.evaluate_property(system_json, gold_json)`

(make sure to replace the system_json and the gold_json with the concrete dictionaries you are comparing, depending on the property and the method)

Now that we have stored the gold values for both properties in our dictionaries `gold_birthyears` and `gold_manufacturers`, and written the evaluation function, we need to obtain the system output as well and then perform evaluation.

For this purpose, we will run our extractors on texts about the same 50 people and 50 devices from Wikipedia. As in the explanation notebook, we will use the `Wikipedia` library for this purpose. Same as in the explanation notebook, we will only process the first three sentences.

In exercises 3a and 3b, we will run all our four processing functions and store the results in four different dictionaries. 
Then, in exercise 3c, we will run the evaluation function four times to compute precision, recall, and F1-score for all four functions.

**Exercise 3a** Run your two extractors about birth years of people (from exercise 1a and 1b) on all 50 documents about people. Save the extracted values in two different dictionaries: `birthyear_regex` and `birthyear_dep`. (3 points)

In [30]:
# Your code here...

**Exercise 3b** Run your extractors about manufacturers of devices (from exercise 2a and 2b) on all 50 documents about devices. Make sure you only process the first three sentences from each document. Save the extracted values in two lists: `manufacturers_regex` and `manufacturers_dep`. (3 points)

In [ ]:
# Your code here...

**Exercise 3c** Run the evaluation function `evaluate_property` to compute the performance for each of your four functions. Print the precision, recall, and F1-scores. (2 points)

In [26]:
# Your code here...

### 4. Reflection (8 points)

For each entity, we will now compare the two methods to extract properties in terms of precision and recall.

**Question 4a** Comparing the precision between the methods based on regular expressions and on syntax dependencies:
* Which method yields lower precision?
* Why do you think this is the case?
* Give an example to support your argument.

(4 points)

In [ ]:
# Your answer here...

**Question 4b** Let's compare the recall for both properties. 
* Which method yields lower recall?
* Why do you think this is the case?
* Give an example to support your argument.

(4 points)

In [13]:
# Your answer here...